# How to work and load MuST-C dataset on your notebooke



> Version 4 is more faster on laoding data into dataframe 

In [ ]:
# !pip install librosa # for working with audio files

In [1]:
import librosa 
from IPython.display import Audio
from tqdm import tqdm
import yaml
from yaml import CLoader
import os
from os.path import join
import pandas as pd

# 1.0 Helper functions for loading data

In [2]:
def get_yaml_data(path):
    with open(path) as f:
        data = yaml.load(f, Loader=CLoader)
        return data                            
    
def get_text_data(path):
    with open(path, 'rt',encoding='utf-8', errors='ignors') as f:
        data = f.readlines()
        return data                           

In [3]:
DataPath   = '/kaggle/input/must-c-en-ar/'
data_split = os.listdir(DataPath)
data_split.remove('docs')
print(data_split)

['tst-HE', 'tst-COMMON', 'train', 'dev']


In [4]:
def create_split_data(split_name):
    split_name  = split_name.strip()
    assert split_name in data_split, f"{split_name} folder dosen't exist on this dataset, only {', '.join(data_split)} are exist"
    split_path = join(DataPath, split_name)
    
    data = get_yaml_data(join(split_path, 'txt', f'{split_name}.yaml'))
    data = pd.DataFrame.from_records(data)
    
    if split_name == 'train':
        
        data['en_text']    = get_text_data(join(split_path, 'txt', f'{split_name}_en.txt'))
        data['ar_text']    = get_text_data(join(split_path, 'txt', f'{split_name}_ar.txt'))
    else:
        data['en_text']    = get_text_data(join(split_path, 'txt', f'{split_name}.en'))
        data['ar_text']    = get_text_data(join(split_path, 'txt', f'{split_name}.ar'))
    
    data = data.sample(frac=1).reset_index(drop=True)                     # shuffle data before return it
    return data

In [5]:
# Test it working
data = create_split_data('dev')
# data

## 2.0 Load data 

In [6]:
print('dev set')
dev = create_split_data('dev')

print('\ntrain set')
train = create_split_data('train')

print('\ntst_COMMON set')
tst_COMMON = create_split_data('tst-COMMON')

print('\ntst-HE set')
tst_HE = create_split_data('tst-HE')



dev set

train set

tst_COMMON set

tst-HE set


In [7]:
dev.describe()

,duration,offset
count,1073.000000,1073.000000
mean,8.308080,561.010065
std,9.014901,381.847271
min,0.050000,12.219999
25%,3.119999,249.099999
50%,5.640000,493.490000
75%,10.200000,857.530000
max,99.590000,1652.989999


In [8]:
train.describe()

,duration,offset
count,212085.000000,212085.000000
mean,7.863565,475.837546
std,8.729069,331.731946
min,0.050000,0.170000
25%,3.000001,210.340000
50%,5.660000,426.639999
75%,9.910000,689.470000
max,604.300000,3603.690000


In [9]:
tst_COMMON.describe()

,duration,offset
count,2019.000000,2019.000000
mean,7.270788,459.836478
std,7.634058,331.908562
min,0.050000,4.900000
25%,2.720000,179.110000
50%,5.020000,383.550000
75%,9.010001,704.800000
max,67.530000,1394.340000


In [16]:
tst_HE.describe()

,duration,offset
count,578.000000,578.000000
mean,9.017024,277.295744
std,7.575855,175.982861
min,0.210000,12.099999
25%,3.640000,134.245000
50%,6.530000,261.319999
75%,11.760000,391.802500
max,52.180000,716.290000


# 3.0 Show sample

In [14]:
duration, offset, speaker_id, wav, en, ar = train.iloc[150_000]
print("en text:", en)
print()
print('ar text', ar)
print()
wave, sr = librosa.load(join(DataPath, 'train','wav', wav), sr=8000, offset=offset, duration=duration)
Audio(wave, rate=sr)

en text: She has amazing land speed for a woman her age, too. Before I know it, she has skiddled across the parking lot and in between the cars, and people behind me, with that kind of usual religious charity that the holidays bring us, wah-wah wah-wah. "I'm coming." Italian hand signals follow. I scoot over. I close the door. I leave the phone books. This is new and fast, just so you — are you still with us?


ar text كان لها أيضا سرعة جديرة بلفت النظر لأمراه سبعينية قبل أن استوعب المشهد كاملا كانت قد انزلقت بعيدا بين أماكن الركن وبين السيارات, في حين أن الأشخاص الذين أحجزهم خلفي بالسيارة يتبعونني بما يشبه التحية الدينية المتعصبة المعتادة التي تمنحها لنا الإجازات (صوت بوق السيارات) أشير إليهم أن صبرا بإشارات إيطالية وبأنني سوف أتحرك فليهدأوا وأسرعت إلى مكانها وأغلقت الباب تاركه أدلة الهاتف أعرف أن هذا إيقاع حكى سريع أمازلتم معي?




In [17]:
# the same wave but defferent sample rate to show the deffernce
wave, sr = librosa.load(join(DataPath, 'train','wav', wav), sr=22024, offset=offset, duration=duration)
Audio(wave, rate=sr)